In [2]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [7]:
df = pd.read_csv("train.csv")
subset = df.sample(n=1000, random_state=42)
subset.to_csv("train_subset.csv", index=False)

In [9]:
import os
import random
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# Определение алфавита (пример: пробел, неизвестный токен и далее символы)
# -----------------------------
alphabet = [' ', '#', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
            'А', 'Б', 'В', 'Г', 'Д', 'Е', 'Ж', 'З', 'И', 'Й', 'К', 'Л',
            'М', 'Н', 'О', 'П', 'Р', 'С', 'Т', 'У', 'Ф', 'Х', 'Ц', 'Ч',
            'Ш', 'Щ', 'Ъ', 'Ы', 'Ь', 'Э', 'Ю', 'Я']
num_classes = len(alphabet)
char_to_idx = {char: idx for idx, char in enumerate(alphabet)}
label_map = {idx: char for idx, char in enumerate(alphabet)}

# -----------------------------
# Аугментация аудио
# -----------------------------
def add_noise(data, noise_factor=0.005):
    noise = np.random.randn(len(data))
    return data + noise_factor * noise

def shift_time(data, shift_max=0.2, shift_direction='both'):
    shift = np.random.randint(int(len(data) * shift_max))
    direction = random.choice([-1, 1]) if shift_direction == 'both' else (1 if shift_direction == 'right' else -1)
    return np.roll(data, shift * direction)

# Функция change_speed оставляем отключённой или можно реализовать через pyrubberband
def change_speed(data, sr, speed_factor=1.2):
    # Временно не используем для упрощения
    return data

# -----------------------------
# SpecAugment для MFCC
# -----------------------------
def spec_augment(mfcc, freq_mask_param=5, time_mask_param=10, num_freq_masks=1, num_time_masks=1):
    augmented = mfcc.copy()
    T, F = augmented.shape
    for _ in range(num_freq_masks):
        f = np.random.randint(0, freq_mask_param)
        f0 = np.random.randint(0, max(F - f, 1))
        augmented[:, f0:f0+f] = 0
    for _ in range(num_time_masks):
        t = np.random.randint(0, time_mask_param)
        t0 = np.random.randint(0, max(T - t, 1))
        augmented[t0:t0+t, :] = 0
    return augmented

# -----------------------------
# Извлечение признаков с Librosa (возвращает последовательность MFCC)
# -----------------------------
def extract_features(file_path, sr=22050, n_mfcc=20, augment=False):
    try:
        audio, sr = librosa.load(file_path, sr=sr)
        audio, _ = librosa.effects.trim(audio)

        if augment:
            if random.random() < 0.3:
                audio = add_noise(audio)
            if random.random() < 0.3:
                audio = shift_time(audio)
            # Если понадобится менять скорость, раскомментируйте:
            # if random.random() < 0.3:
            #     speed_factor = random.uniform(0.8, 1.2)
            #     audio = change_speed(audio, sr, speed_factor)

        # Извлечение MFCC; получаем форму (T, n_mfcc)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc).T

        if augment:
            mfcc = spec_augment(mfcc)

        return mfcc
    except Exception as e:
        print(f"Ошибка при обработке файла {file_path}: {e}")
        return np.zeros((1, n_mfcc))

# -----------------------------
# Преобразование строки метки в последовательность индексов
# -----------------------------
def string_to_indices(raw_string):
    indices = []
    for ch in str(raw_string):
        if ch in char_to_idx:
            indices.append(char_to_idx[ch])
        else:
            indices.append(char_to_idx['#'])
    return indices

# -----------------------------
# PyTorch Dataset (ожидается, что CSV содержит столбцы "id" и "message")
# -----------------------------
class MorseDataset(Dataset):
    def __init__(self, csv_file, data_dir, sr=22050, n_mfcc=20, augment=True, limit=None):
        self.data_frame = pd.read_csv(csv_file)
        if limit is not None:
            self.data_frame = self.data_frame.sample(n=limit, random_state=42).reset_index(drop=True)
        self.data_dir = data_dir
        self.sr = sr
        self.n_mfcc = n_mfcc
        self.augment = augment

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        file_path_in_csv = self.data_frame.iloc[idx]['id']
        raw_label = self.data_frame.iloc[idx]['message']
        label_indices = string_to_indices(raw_label)
        label = torch.tensor(label_indices, dtype=torch.long)

        full_file_path = os.path.join(self.data_dir, file_path_in_csv)
        features = extract_features(full_file_path, sr=self.sr, n_mfcc=self.n_mfcc, augment=self.augment)
        features = torch.tensor(features, dtype=torch.float)

        # Сохраняем оригинальное сообщение для валидации
        return features, label, raw_label

# -----------------------------
# Collate-функция для переменных длин
# -----------------------------
def collate_fn(batch):
    features_list, labels_list, raw_labels = zip(*batch)
    feature_lengths = [f.shape[0] for f in features_list]
    max_time = max(feature_lengths)
    padded_features = []
    for f in features_list:
        pad_size = max_time - f.shape[0]
        if pad_size > 0:
            pad = torch.zeros(pad_size, f.shape[1])
            padded_f = torch.cat([f, pad], dim=0)
        else:
            padded_f = f
        padded_features.append(padded_f)
    padded_features = torch.stack(padded_features)  # (batch, max_time, n_mfcc)

    label_lengths = [len(lbl) for lbl in labels_list]
    concatenated_labels = torch.cat(labels_list)

    return padded_features, concatenated_labels, feature_lengths, label_lengths, raw_labels

# -----------------------------
# Улучшенная модель: 1D-CNN + BiLSTM
# -----------------------------
class MorseClassifier(nn.Module):
    def __init__(self, input_dim, cnn_out_channels, cnn_kernel_size, hidden_dim, num_layers, num_classes):
        super(MorseClassifier, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=cnn_out_channels, kernel_size=cnn_kernel_size, padding=cnn_kernel_size//2),
            nn.BatchNorm1d(cnn_out_channels),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)
        )
        self.lstm = nn.LSTM(input_size=cnn_out_channels, hidden_size=hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        # x: (batch, T, input_dim) -> для CNN: (batch, input_dim, T)
        x = x.transpose(1, 2)
        cnn_out = self.cnn(x)  # (batch, cnn_out_channels, T')
        cnn_out = cnn_out.transpose(1, 2)  # (batch, T', cnn_out_channels)
        lstm_out, _ = self.lstm(cnn_out)      # (batch, T', hidden_dim*2)
        outputs = self.fc(lstm_out)           # (batch, T', num_classes)
        return outputs

# -----------------------------
# Функция beam search для CTC-декодирования
# -----------------------------
from collections import defaultdict
import heapq

def ctc_beam_search(log_probs, beam_width=10, blank=0):
    T, C = log_probs.shape
    beam = [(0.0, [])]  # (log_prob, seq)
    for t in range(T):
        new_beam = defaultdict(lambda: -float("inf"))
        for score, seq in beam:
            for c in range(C):
                new_seq = seq + [c]
                new_score = score + log_probs[t, c].item()
                new_beam[tuple(new_seq)] = max(new_beam[tuple(new_seq)], new_score)
        beam = heapq.nlargest(beam_width, [(s, list(k)) for k, s in new_beam.items()])
    best_seq = beam[0][1]
    decoded = []
    prev = None
    for c in best_seq:
        if c != blank and c != prev:
            decoded.append(c)
        prev = c
    return decoded

# -----------------------------
# Функция вычисления расстояния Левенштейна
# -----------------------------
def levenshtein(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1):
        dp[i][0] = i
    for j in range(n+1):
        dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + cost)
    return dp[m][n]

# -----------------------------
# Тренировочный цикл с CTCLoss и сохранением loss для графика
# -----------------------------
def train_model(model, dataloader, criterion, optimizer, num_epochs=25, device='cpu'):
    model.to(device)
    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for features, concatenated_labels, feature_lengths, label_lengths, _ in tqdm(dataloader, desc=f"Эпоха {epoch+1}", leave=False):
            features = features.to(device)
            concatenated_labels = concatenated_labels.to(device)

            # Forward
            optimizer.zero_grad()
            outputs = model(features)  # (batch, T', num_classes)
            outputs = outputs.permute(1, 0, 2)  # (T', batch, num_classes)
            log_probs = nn.functional.log_softmax(outputs, dim=2)

            input_lengths = torch.full(
                size=(features.size(0),),
                fill_value=log_probs.size(0),  # T'
                dtype=torch.long
            )
            target_lengths = torch.tensor(label_lengths, dtype=torch.long)

            # 🔐 CTC ограничение: input_length >= 2 * target_length - 1
            if any(input_lengths[i] < 2 * target_lengths[i] - 1 for i in range(len(input_lengths))):
                print(f"[!] Пропущен батч: input_lengths={input_lengths.tolist()}, target_lengths={target_lengths.tolist()}")
                continue

            # Loss & Backprop
            loss = criterion(log_probs, concatenated_labels, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / max(len(dataloader), 1)
        epoch_losses.append(avg_loss)
        print(f"Эпоха {epoch+1}/{num_epochs} — Потеря: {avg_loss:.4f}")

    return model, epoch_losses


# -----------------------------
# Функция валидации (с вычислением среднего расстояния Левенштейна)
# -----------------------------
def validate_model(model, dataset, device='cpu', blank=0):
    model.eval()
    model.to(device)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    total_dist = 0
    count = 0
    for features, _, feature_lengths, _, raw_labels in tqdm(dataloader, desc="Валидация"):
        features = features.to(device)
        outputs = model(features)  # (batch=1, T', num_classes)
        outputs = outputs.squeeze(0)  # (T', num_classes)
        log_probs = nn.functional.log_softmax(outputs, dim=1)
        pred_indices = ctc_beam_search(log_probs, beam_width=10, blank=blank)
        if len(pred_indices) == 0:
            pred_str = "#"
        else:
            pred_str = ''.join([label_map[idx] for idx in pred_indices])
        true_str = raw_labels[0]  # так как batch=1
        dist = levenshtein(pred_str, true_str)
        total_dist += dist
        count += 1
    avg_distance = total_dist / count if count > 0 else None
    print(f"Среднее расстояние Левенштейна по валидации: {avg_distance:.2f}")
    return avg_distance

# -----------------------------
# Функция предсказания и создания submission.csv с beam search
# -----------------------------
def predict_and_submit(model, test_csv, data_dir, submission_file='submission.csv',
                       sr=22050, n_mfcc=20, device='cpu', blank=0):
    model.eval()
    model.to(device)
    test_df = pd.read_csv(test_csv)
    predictions = []
    for i in tqdm(range(len(test_df)), desc="Предсказание"):
        file_path_in_csv = test_df.loc[i, 'id']
        full_file_path = os.path.join(data_dir, file_path_in_csv)
        feats = extract_features(full_file_path, sr=sr, n_mfcc=n_mfcc, augment=False)
        feats_tensor = torch.tensor(feats, dtype=torch.float).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(feats_tensor)
            outputs = outputs.squeeze(0)  # (T', num_classes)
            log_probs = nn.functional.log_softmax(outputs, dim=1)
            pred_indices = ctc_beam_search(log_probs, beam_width=10, blank=blank)
        if len(pred_indices) == 0:
            pred_str = "#"
        else:
            pred_str = ''.join([label_map[idx] for idx in pred_indices])
        predictions.append([file_path_in_csv, pred_str])
    submission_df = pd.DataFrame(predictions, columns=['id', 'message'])
    submission_df.to_csv(submission_file, index=False)
    print(f"Файл {submission_file} успешно сохранён!")

# -----------------------------
# Основная функция
# -----------------------------
def main():
    # Пути к файлам согласно черновику
    train_csv = 'train.csv'   # Файл с "id" и "message"
    test_csv = 'test.csv'     # Файл с "id"
    data_dir = 'morse_dataset/morse_dataset/'  # Папка с аудиофайлами
    submission_file = 'submission.csv'

    sr = 22050
    n_mfcc = 20
    batch_size = 16
    num_epochs = 5
    learning_rate = 0.001

    # Можно использовать часть датасета для быстрой отладки, например, limit=500
    full_dataset = MorseDataset(train_csv, data_dir, sr=sr, n_mfcc=n_mfcc, augment=True)
    # Разобьём на train и validation (80/20)
    total_samples = len(full_dataset)
    indices = list(range(total_samples))
    split = int(0.8 * total_samples)
    train_indices = indices[:split]
    val_indices = indices[split:]

    from torch.utils.data import Subset
    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    # Инициализируем улучшенную модель: параметры CNN, LSTM
    cnn_out_channels = 64
    cnn_kernel_size = 3
    hidden_dim = 128
    num_layers = 2
    model = MorseClassifier(input_dim=n_mfcc, cnn_out_channels=cnn_out_channels,
                            cnn_kernel_size=cnn_kernel_size, hidden_dim=hidden_dim,
                            num_layers=num_layers, num_classes=num_classes)

    # Используем CTCLoss (blank=0)
    criterion = nn.CTCLoss(blank=0)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Используем устройство: {device}")

    # Обучение и сбор истории loss
    model, epoch_losses = train_model(model, train_loader, criterion, optimizer, num_epochs=num_epochs, device=device)

    # Строим график loss по эпохам
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, num_epochs+1), epoch_losses, marker='o')
    plt.title('Loss по эпохам')
    plt.xlabel('Эпоха')
    plt.ylabel('Средняя потеря')
    plt.grid(True)
    plt.show()

    # Валидация: вычисляем среднее расстояние Левенштейна
    print("Валидация модели:")
    validate_model(model, val_dataset, device=device, blank=0)

    # Предсказание для тестового набора и сохранение submission.csv
    predict_and_submit(model, test_csv, data_dir, submission_file=submission_file,
                       sr=sr, n_mfcc=n_mfcc, device=device, blank=0)

if __name__ == '__main__':
    main()


Используем устройство: cpu


Эпоха 1:   0%|          | 0/1500 [00:00<?, ?it/s]

Эпоха 1/5 — Потеря: 3.8074


Эпоха 2:   0%|          | 0/1500 [00:00<?, ?it/s]

Эпоха 2/5 — Потеря: 3.5268


Эпоха 3:   0%|          | 0/1500 [00:00<?, ?it/s]

KeyboardInterrupt: 